In [2]:
from math import sumprod
from typing import NewType

import torch
from sympy.stats.rv import probability
from tornado.concurrent import run_on_executor

print(torch.backends.mps.is_available())

ERROR! Session/line number was not unique in database. History logging moved to new session 27
True


In [3]:
tensor0D=torch.tensor(1)
print(tensor0D)

tensor1D=torch.tensor([1,2,3])
print(tensor1D)

tensor2D=torch.tensor([[1,12],[3,4]])
print(tensor2D)

tensor3D=torch.tensor([[[1,2],[3,4]],[[5,6],[7,8]]])
print(tensor3D)

tensor(1)
tensor([1, 2, 3])
tensor([[ 1, 12],
        [ 3,  4]])
tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])


In [4]:
print(tensor1D.dtype)

floatvec=tensor1D.to(torch.float32)
print(floatvec.dtype)

torch.int64
torch.float32


In [5]:
print(tensor2D.shape)
print(tensor3D.shape)

print(tensor2D.matmul(tensor2D))
print(tensor3D.matmul(tensor3D))

print(tensor3D @ tensor3D)

torch.Size([2, 2])
torch.Size([2, 2, 2])
tensor([[37, 60],
        [15, 52]])
tensor([[[  7,  10],
         [ 15,  22]],

        [[ 67,  78],
         [ 91, 106]]])
tensor([[[  7,  10],
         [ 15,  22]],

        [[ 67,  78],
         [ 91, 106]]])


In [6]:
import torch.nn.functional as F

y=torch.tensor([1.0])
x1=torch.tensor([1.1])
w1=torch.tensor([2.2])
b=torch.tensor([0.0])
z=x1*w1+b

a=torch.sigmoid(z) #output
loss=F.binary_cross_entropy(a,y)
print("y= ",y)
print("x1= ",x1)
print("w1= ",w1)
print("b= ",b)
print("z= ",z)
print("a= ",a)
print("loss= ",loss)


y=  tensor([1.])
x1=  tensor([1.1000])
w1=  tensor([2.2000])
b=  tensor([0.])
z=  tensor([2.4200])
a=  tensor([0.9183])
loss=  tensor(0.0852)


In [7]:
from torch.autograd import grad

y=torch.tensor([1.0])
x1=torch.tensor([1.1])
w1=torch.tensor([2.2],requires_grad=True)
b=torch.tensor([0.0],requires_grad=True)

z=x1*w1+b
a=torch.sigmoid(z)
loss=F.binary_cross_entropy(a,y)

# grad_L_w1=grad(loss,w1,retain_graph=True) #partial derivative of the loss with respect to w1 (how much does the loss change
#                                           # and in what direction if I change this weight )
# grad_L_b=grad(loss,b,retain_graph=True)

loss.backward()
print(w1.grad)
print(b.grad)

print(grad_L_w1,grad_L_b)

tensor([-0.0898])
tensor([-0.0817])


NameError: name 'grad_L_w1' is not defined

In [8]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self,inputs,outputs):
        super().__init__()

        self.layers=torch.nn.Sequential(
            torch.nn.Linear(inputs,30),
            torch.nn.ReLU(),

            torch.nn.Linear(30,20),
            torch.nn.ReLU(),

            torch.nn.Linear(20,outputs),
        )
    def forward(self,x):
        return self.layers(x)

model=NeuralNetwork(50,3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [9]:
num_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total nr of trainable parameters: ",num_params)

print(model.layers[0].weight)
print(model.layers[0].weight.shape)

Total nr of trainable parameters:  2213
Parameter containing:
tensor([[ 0.1178,  0.1057, -0.1408,  ...,  0.1037,  0.1405, -0.0332],
        [ 0.1377, -0.0495,  0.0825,  ...,  0.0068,  0.1291, -0.0487],
        [ 0.0937, -0.1068,  0.0851,  ..., -0.1407,  0.1021,  0.1018],
        ...,
        [ 0.1201, -0.0063,  0.0417,  ...,  0.0829,  0.0985,  0.1336],
        [ 0.0951, -0.1375, -0.0750,  ..., -0.1255, -0.1083,  0.1223],
        [ 0.1379, -0.0756,  0.0839,  ..., -0.1294,  0.1171, -0.0764]],
       requires_grad=True)
torch.Size([30, 50])


In [10]:
torch.manual_seed(123)
model=NeuralNetwork(50,3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [11]:
torch.manual_seed(123)
X=torch.rand((1,50))
print(X)
out=model(X)
print(out)

tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025, 0.1841,
         0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274,
         0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745, 0.6584, 0.2775,
         0.8573, 0.8993, 0.0390, 0.9268, 0.7388, 0.7179, 0.7058, 0.9156, 0.4340,
         0.0772, 0.3565, 0.1479, 0.5331, 0.4066, 0.2318, 0.4545, 0.9737, 0.4606,
         0.5159, 0.4220, 0.5786, 0.9455, 0.8057]])
tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [12]:
X_train=torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train=torch.tensor([0,0,0,1,1])

X_test=torch.tensor([
    [-0.8, 2.8],
[2.6, -1.6],
])
y_test=torch.tensor([0,1])

In [13]:
from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    def __init__(self,X,y):
        self.features=X
        self.labels=y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds=ToyDataset(X_train,y_train)
test_ds=ToyDataset(X_test,y_test)

print(train_ds[1])
print(test_ds[1])

(tensor([-0.9000,  2.9000]), tensor(0))
(tensor([ 2.6000, -1.6000]), tensor(1))


In [14]:
train_loader=DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

test_loader=DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [15]:
torch.manual_seed(123)
modelNN=NeuralNetwork(2,2)

optimizer=torch.optim.SGD(modelNN.parameters(),lr=0.5)

num_epochs=3
for epoch in range(num_epochs):
    modelNN.train()

    for batch_idx, (features,labels) in enumerate(train_loader):
        logits=modelNN(features)

        loss=F.cross_entropy(logits,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch}, weights are {model.layers[0].weight}")

        print(f"For epoch {epoch+1}/{num_epochs}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")


Epoch 0, weights are Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)
For epoch 1/3 | Batch 000/002 | Train Loss: 0.75
Epoch 0, weights are Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,

In [16]:
modelNN.eval()
with torch.no_grad():
    output=modelNN(X_train)
print(output)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [17]:
torch.set_printoptions(sci_mode=False)
probabilities=torch.softmax(output,dim=1)
print(probabilities)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


In [18]:
torch.argmax(probabilities,dim=1)

tensor([0, 0, 0, 1, 1])